# 02 · Data Preparation

Building the final table: one row per team per season. Decisions and reasons go in `docs/project_notes.md`.

In [ ]:
import sys
from pathlib import Path
sys.path.append("..")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loading import load_all

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")
FIG = Path("../figures"); FIG.mkdir(exist_ok=True)   # plots for the slides are saved here

tables = load_all()

## Team table (task a)

In [ ]:
# Which data we use. Change these two lines if we decide otherwise.
LEAGUES = ["NHL"]
FIRST_YEAR = 1979

teams = tables["Teams"]
nhl = teams[teams["lgID"].isin(LEAGUES)].copy()

# Results per game, because seasons had different lengths
nhl["pts_per_game"] = nhl["Pts"] / nhl["G"]
nhl["gf_per_game"] = nhl["GF"] / nhl["G"]
nhl["ga_per_game"] = nhl["GA"] / nhl["G"]

# Compared to that season's league average (1.0 = average), because scoring levels changed over the years
# and the points system changed (overtime-loss point from 1999, no ties and shootouts from 2005)
nhl["pts_vs_avg"] = nhl["pts_per_game"] / nhl.groupby("year")["pts_per_game"].transform("mean")
nhl["gf_vs_avg"] = nhl["gf_per_game"] / nhl.groupby("year")["gf_per_game"].transform("mean")
nhl["ga_vs_avg"] = nhl["ga_per_game"] / nhl.groupby("year")["ga_per_game"].transform("mean")

nhl["made_playoffs"] = nhl["playoff"].notna().astype(int)

# How many teams were in the same division that season
nhl["div_size"] = nhl.groupby(["year", "divID"])["tmID"].transform("count")

# Last season's results for the same club (franchID follows a club when it moves city)
nhl = nhl.sort_values(["franchID", "year"])
for col in ["pts_vs_avg", "gf_vs_avg", "ga_vs_avg", "made_playoffs", "rank"]:
    nhl["last_" + col] = nhl.groupby("franchID")[col].shift(1)
nhl["new_team"] = nhl["last_rank"].isna().astype(int)

team_table = nhl[nhl["year"] >= FIRST_YEAR][[
    "year", "franchID", "tmID", "name", "divID", "div_size",
    "rank",
    "last_pts_vs_avg", "last_gf_vs_avg", "last_ga_vs_avg",
    "last_made_playoffs", "last_rank", "new_team",
]].reset_index(drop=True)

print(team_table.shape)
team_table.head()

In [ ]:
Path("../data").mkdir(exist_ok=True)
team_table.to_csv("../data/team_table.csv", index=False)

## Roster information (task a)

In [ ]:
scoring = tables["Scoring"]
goalies = tables["Goalies"]

# For each player on a roster we use his most recent season before this one.
# Usually that is last season; for players who missed seasons (injury, minor leagues)
# it is the last season they played; for 2005 it is 2003 (2004 lockout).

# --- Skaters: points the roster scored in their most recent season ---
# Players on the team at the start of the season = stint 1 for that team and season
roster = scoring[scoring["stint"] == 1][["playerID", "year", "tmID"]]
roster = roster[roster["year"] >= FIRST_YEAR].sort_values("year")

# Each player's total points per season (all teams he played for)
player_seasons = scoring.groupby(["playerID", "year"])["Pts"].sum().reset_index()
player_seasons = player_seasons.rename(columns={"year": "played_year", "Pts": "last_pts"}).sort_values("played_year")

# Match each roster player to his most recent season before this one
roster = pd.merge_asof(roster, player_seasons, left_on="year", right_on="played_year",
                       by="playerID", allow_exact_matches=False)
roster_pts = roster.groupby(["year", "tmID"])["last_pts"].sum().reset_index()   # rookies count as 0

# Compare to that season's average team, because scoring levels changed
roster_pts["roster_pts_vs_avg"] = roster_pts["last_pts"] / roster_pts.groupby("year")["last_pts"].transform("mean")

# --- Goalies: goals they conceded per 60 minutes in their most recent season ---
g_roster = goalies[goalies["stint"] == 1][["playerID", "year", "tmID"]]
g_roster = g_roster[g_roster["year"] >= FIRST_YEAR].sort_values("year")

goalie_seasons = goalies.groupby(["playerID", "year"])[["GA", "Min"]].sum().reset_index()
goalie_seasons = goalie_seasons.rename(columns={"year": "played_year", "GA": "last_GA", "Min": "last_Min"}).sort_values("played_year")

g_roster = pd.merge_asof(g_roster, goalie_seasons, left_on="year", right_on="played_year",
                         by="playerID", allow_exact_matches=False)
g_roster = g_roster.dropna(subset=["last_Min"])   # goalies with no earlier season are left out
g_team = g_roster.groupby(["year", "tmID"])[["last_GA", "last_Min"]].sum().reset_index()
g_team["goalie_ga_per_60"] = g_team["last_GA"] / g_team["last_Min"] * 60
g_team["roster_goalie_ga_vs_avg"] = g_team["goalie_ga_per_60"] / g_team.groupby("year")["goalie_ga_per_60"].transform("mean")

# --- Add both to the team table ---
team_table = team_table.drop(columns=["roster_pts_vs_avg", "roster_goalie_ga_vs_avg"], errors="ignore")
team_table = team_table.merge(roster_pts[["year", "tmID", "roster_pts_vs_avg"]], on=["year", "tmID"], how="left")
team_table = team_table.merge(g_team[["year", "tmID", "roster_goalie_ga_vs_avg"]], on=["year", "tmID"], how="left")

# Save the updated table
team_table.to_csv("../data/team_table.csv", index=False)

print(team_table.shape)
team_table[["year", "name", "rank", "last_pts_vs_avg", "roster_pts_vs_avg", "roster_goalie_ga_vs_avg"]].head()

## Coach information (task b)

## Data quality, missing values and outliers